# BCEDF: Breast Cancer Early Detection Framework
## Thesis Implementation — TensorFlow/Keras

**Architectures:** Custom CNN, ResNet50, DenseNet121  
**Datasets:** MIAS, BreakHis, INbreast (individual + combined)  
**Reference:** “Final Thesis Chetna” — Tables 6.1, 8.1, 8.2

**Setup:**
1. Add dataset inputs: `mias`, `breakhis`, `inbreast`
2. Settings → Accelerator → **GPU P100/T4**, Internet: **ON**
3. Notebook clones code from: https://github.com/Chetna7860/thesis

In [ ]:
# Cell 1: Clone repo from GitHub + install dependencies
!rm -rf /kaggle/working/thesis
!git clone https://github.com/Chetna7860/thesis.git /kaggle/working/thesis
%cd /kaggle/working/thesis
!pip install -q numpy pandas scikit-learn scipy matplotlib seaborn opencv-python tqdm pydicom xlrd --no-build-isolation
print('Repo cloned and dependencies installed')

In [ ]:
# Cell 2: GPU verification
import tensorflow as tf
print(f'TensorFlow: {tf.__version__}')
gpus = tf.config.list_physical_devices('GPU')
print(f'CUDA available: {len(gpus) > 0}')
if gpus:
    for gpu in gpus:
        print(f'GPU: {gpu}')
else:
    print('WARNING: No GPU detected — training will be very slow')

In [ ]:
# Cell 3: Paths, imports, and config
import os, sys, json, datetime, warnings, random, time as time_module
sys.path.insert(0, '/kaggle/working/thesis')
os.chdir('/kaggle/working/thesis')
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, models, applications, optimizers, losses, callbacks, preprocessing
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             roc_auc_score, confusion_matrix, roc_curve)
from sklearn.model_selection import train_test_split
import cv2
import pydicom
import xlrd
warnings.filterwarnings('ignore')

# Dataset paths (Kaggle inputs)
DATASET_PATHS = {
    'mias': '/kaggle/input/mias',
    'breakhis': '/kaggle/input/breakhis/BreaKHis_v1/BreaKHis_v1/histology_slides/breast',
    'inbreast': '/kaggle/input/inbreast'
}

# Output directory
OUTPUT_DIR = '/kaggle/working/outputs'
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Training config (thesis §7)
EPOCHS = 50
BATCH_SIZE = 32
LR_PHASE1 = 1e-3
LR_PHASE2 = 1e-5
IMG_SIZE = 224
NUM_CLASSES = 2
PHASE1_EPOCHS = 10
PATIENCE = 10
ARCHITECTURES = ['cnn', 'resnet50', 'densenet121']
DATASET_CONFIGS = [['mias'], ['breakhis'], ['inbreast'], ['mias', 'breakhis', 'inbreast']]

# Grad-CAM layer names (thesis §6.5)
GRAD_CAM_LAYERS = {
    'cnn': 'conv2d_2',
    'resnet50': 'conv5_block3_out',
    'densenet121': 'conv5_block16_concat'
}

print('Config loaded')
print(f'Architectures: {ARCHITECTURES}')
print(f'Dataset configs: {DATASET_CONFIGS}')
print(f'Epochs: {EPOCHS}, Batch: {BATCH_SIZE}, LR: {LR_PHASE1}/{LR_PHASE2}')

In [ ]:
# Cell 4: Dataset loaders (auto-discover paths recursively)
def find_file(root, target):
    for dirpath, _, files in os.walk(root):
        if target in files:
            return os.path.join(dirpath, target)
    return None

def find_dir(root, target):
    for dirpath, dirs, _ in os.walk(root):
        if target in dirs:
            return os.path.join(dirpath, target)
    return None

def load_mias(path):
    """MIAS: 322 PGMs + Info.txt. Label: any M annotation -> malignant."""
    images, labels = [], []
    info_path = find_file(path, 'Info.txt')
    if info_path is None:
        print('ERROR: Info.txt not found under', path)
        return np.array([]), np.array([])
    with open(info_path, 'r') as f:
        lines = f.readlines()
    label_map = {}
    for line in lines[1:]:
        parts = line.strip().split()
        ref = parts[0]
        cls = parts[2]
        if cls == 'NORM':
            label_map.setdefault(ref, 0)
        else:
            sev = parts[3] if len(parts) >= 4 else ''
            if sev == 'M':
                label_map[ref] = 1
            elif ref not in label_map:
                label_map[ref] = 0 if sev == 'B' else 0
    for ref, label in label_map.items():
        p = find_file(path, f'{ref}.pgm')
        if p is not None:
            img = cv2.imread(p, cv2.IMREAD_GRAYSCALE)
            if img is not None:
                images.append(img)
                labels.append(label)
    print(f'MIAS: {len(images)} images ({sum(1 for l in labels if l==0)} benign, {sum(labels)} malignant)')
    return np.array(images), np.array(labels)


def load_breakhis(path):
    """BreakHis: 7909 PNGs from folder structure."""
    images, labels = [], []
    for cls_name, label in [('benign', 0), ('malignant', 1)]:
        cls_dir = find_dir(path, cls_name)
        if cls_dir is None:
            continue
        for root, _, files in os.walk(cls_dir):
            for f in files:
                if f.endswith('.png'):
                    img_path = os.path.join(root, f)
                    img = cv2.imread(img_path)
                    if img is not None:
                        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
                        images.append(img)
                        labels.append(label)
    print(f'BreakHis: {len(images)} images ({sum(1 for l in labels if l==0)} benign, {sum(labels)} malignant)')
    return np.array(images), np.array(labels)


def load_inbreast(path):
    """INbreast: 410 DICOMs + XLS. BI-RADS 1-3 -> benign, 4-6 -> malignant."""
    xls_path = find_file(path, 'INbreast.xls')
    if xls_path is None:
        print('ERROR: INbreast.xls not found under', path)
        return np.array([]), np.array([])
    wb = xlrd.open_workbook(xls_path)
    ws = wb.sheet_by_index(0)
    dicom_map = {}
    for r in range(1, ws.nrows):
        fname = str(ws.cell_value(r, 5)).strip().split('.')[0]
        birads = str(ws.cell_value(r, 7)).strip()
        if birads in ['1.0', '2.0', '3.0']:
            dicom_map[fname] = 0
        elif birads in ['4a', '4b', '4c', '5.0', '6.0']:
            dicom_map[fname] = 1
    dcm_dir = find_dir(path, 'ALL-IMGS')
    if dcm_dir is None:
        print('ERROR: ALL-IMGS directory not found under', path)
        return np.array([]), np.array([])
    images, labels = [], []
    for f in sorted(os.listdir(dcm_dir)):
        if f.endswith('.dcm'):
            num_part = f.split('_')[0]
            if num_part in dicom_map:
                try:
                    ds = pydicom.dcmread(os.path.join(dcm_dir, f))
                    img = ds.pixel_array
                    img = ((img - img.min()) / (img.max() - img.min() + 1e-8) * 255).astype(np.uint8)
                    img = cv2.cvtColor(img, cv2.COLOR_GRAY2RGB)
                    images.append(img)
                    labels.append(dicom_map[num_part])
                except Exception as e:
                    print(f'  Warning: could not read {f}: {e}')
    print(f'INbreast: {len(images)} images ({sum(1 for l in labels if l==0)} benign, {sum(labels)} malignant)')
    return np.array(images), np.array(labels)


def get_dataset(name):
    base = DATASET_PATHS[name]
    if not os.path.exists(base):
        base = os.path.join('/kaggle/input', name)
    if name == 'mias':
        return load_mias(base)
    elif name == 'breakhis':
        return load_breakhis(base)
    elif name == 'inbreast':
        return load_inbreast(base)

print('Dataset loaders defined')

In [ ]:
# Cell 5: Preprocessing pipeline (thesis §6.3)
def preprocess_image(img):
    """
    1. Gaussian noise reduction
    2. CLAHE on LAB L-channel (ClipLimit=2.0, TileGrid=8x8)
    3. Resize to 224x224 (bicubic)
    4. Normalize to [0, 1]
    """
    if img.dtype != np.uint8:
        img = (img * 255).astype(np.uint8)
    img = cv2.GaussianBlur(img, (3, 3), 0)
    if len(img.shape) == 2 or img.shape[2] == 1:
        gray = img if len(img.shape) == 2 else img[:,:,0]
        clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
        enhanced = clahe.apply(gray)
        img_rgb = cv2.cvtColor(enhanced, cv2.COLOR_GRAY2RGB)
    else:
        lab = cv2.cvtColor(img, cv2.COLOR_RGB2LAB)
        l, a, b = cv2.split(lab)
        clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
        l_enhanced = clahe.apply(l)
        lab_enhanced = cv2.merge([l_enhanced, a, b])
        img_rgb = cv2.cvtColor(lab_enhanced, cv2.COLOR_LAB2RGB)
    img_rgb = cv2.resize(img_rgb, (IMG_SIZE, IMG_SIZE), interpolation=cv2.INTER_CUBIC)
    img_rgb = img_rgb.astype(np.float32) / 255.0
    return img_rgb


def preprocess_dataset(images, labels, name=''):
    processed = []
    for img in images:
        processed.append(preprocess_image(img))
    print(f'{name}: preprocessed {len(processed)} images')
    return np.array(processed), np.array(labels)


def create_augmentation():
    """Thesis augmentation: rotation=30, zoom=0.2, flip, brightness=[0.8,1.2]."""
    return preprocessing.image.ImageDataGenerator(
        rotation_range=30,
        zoom_range=0.2,
        horizontal_flip=True,
        vertical_flip=True,
        brightness_range=[0.8, 1.2],
        fill_mode='nearest'
    )


def train_val_test_split(images, labels, val_split=0.15, test_split=0.10):
    X_temp, X_test, y_temp, y_test = train_test_split(
        images, labels, test_size=test_split, stratify=labels, random_state=42
    )
    val_frac = val_split / (1 - test_split)
    X_train, X_val, y_train, y_val = train_test_split(
        X_temp, y_temp, test_size=val_frac, stratify=y_temp, random_state=42
    )
    return (X_train, y_train), (X_val, y_val), (X_test, y_test)


def compute_class_weight(labels):
    counts = np.bincount(labels)
    total = len(labels)
    return {i: total / (len(counts) * c) for i, c in enumerate(counts) if c > 0}

print('Preprocessing and augmentation defined')

In [ ]:
# Cell 6: Model builders (thesis §6.4)
def build_cnn():
    """Custom CNN ~2.1M params: 3x ConvBlock(32,64,128) → GAP → Dense(512) → Drop(0.3) → Softmax(2)"""
    model = models.Sequential([
        layers.Input(shape=(IMG_SIZE, IMG_SIZE, 3)),
        layers.Conv2D(32, 3, padding='same', activation='relu'),
        layers.BatchNormalization(),
        layers.MaxPooling2D(2),
        layers.Conv2D(64, 3, padding='same', activation='relu'),
        layers.BatchNormalization(),
        layers.MaxPooling2D(2),
        layers.Conv2D(128, 3, padding='same', activation='relu'),
        layers.BatchNormalization(),
        layers.MaxPooling2D(2),
        layers.GlobalAveragePooling2D(),
        layers.Dense(512, activation='relu'),
        layers.Dropout(0.3),
        layers.Dense(NUM_CLASSES, activation='softmax')
    ])
    return model


def classification_head(input_tensor):
    """Shared head: GAP → BN → Drop(0.3) → Dense(512,ReLU) → Drop(0.2) → Dense(2,Softmax)"""
    x = layers.GlobalAveragePooling2D()(input_tensor)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(0.3)(x)
    x = layers.Dense(512, activation='relu')(x)
    x = layers.Dropout(0.2)(x)
    return layers.Dense(NUM_CLASSES, activation='softmax')(x)


def build_resnet50():
    backbone = applications.ResNet50(
        include_top=False, weights='imagenet', input_shape=(IMG_SIZE, IMG_SIZE, 3)
    )
    backbone.trainable = False
    outputs = classification_head(backbone.output)
    return models.Model(inputs=backbone.input, outputs=outputs), backbone


def build_densenet121():
    backbone = applications.DenseNet121(
        include_top=False, weights='imagenet', input_shape=(IMG_SIZE, IMG_SIZE, 3)
    )
    backbone.trainable = False
    outputs = classification_head(backbone.output)
    return models.Model(inputs=backbone.input, outputs=outputs), backbone


def get_model(name):
    if name == 'cnn':
        return build_cnn(), None
    elif name == 'resnet50':
        return build_resnet50()
    elif name == 'densenet121':
        return build_densenet121()

print('Models defined')

In [ ]:
# Cell 7: Training, evaluation, Grad-CAM (thesis §6.4, §6.5, §7)
def train_model(model_name, dataset_names):
    run_name = f'{model_name}_' + '+'.join(dataset_names)
    run_dir = os.path.join(OUTPUT_DIR, model_name)
    subdirs = {k: os.path.join(run_dir, k) for k in ['checkpoints', 'plots', 'models', 'gradcam']}
    for d in subdirs.values():
        os.makedirs(d, exist_ok=True)
    print(f'\n{"="*70}')
    print(f'  RUN: {run_name}')
    print(f'{"="*70}')

    all_images, all_labels = [], []
    for ds_name in dataset_names:
        print(f'Loading {ds_name}...')
        imgs, lbls = get_dataset(ds_name)
        imgs, lbls = preprocess_dataset(imgs, lbls, ds_name)
        all_images.append(imgs)
        all_labels.append(lbls)
    X = np.concatenate(all_images, axis=0)
    y = np.concatenate(all_labels, axis=0)
    print(f'Total: {len(X)} images ({sum(1 for l in y if l==0)} benign, {sum(y)} malignant)')

    (X_train, y_train), (X_val, y_val), (X_test, y_test) = train_val_test_split(X, y)
    print(f'Train: {len(X_train)} | Val: {len(X_val)} | Test: {len(X_test)}')
    class_weight = compute_class_weight(y_train)
    print(f'Class weights: {class_weight}')

    model, backbone = get_model(model_name)
    model.compile(
        optimizer=optimizers.Adam(learning_rate=LR_PHASE1),
        loss=losses.SparseCategoricalCrossentropy(),
        metrics=['accuracy']
    )
    print(f'{model_name}: {model.count_params():,} params')

    cb = [
        callbacks.EarlyStopping(monitor='val_accuracy', patience=PATIENCE,
                                restore_best_weights=True, verbose=1),
        callbacks.ModelCheckpoint(os.path.join(subdirs['checkpoints'], f'{model_name}_best.h5'),
                                  monitor='val_accuracy', save_best_only=True, verbose=0),
        callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=5, min_lr=1e-7, verbose=1)
    ]
    datagen = create_augmentation()
    train_flow = datagen.flow(X_train, y_train, batch_size=BATCH_SIZE)

    history = {'loss': [], 'accuracy': [], 'val_loss': [], 'val_accuracy': []}

    if model_name != 'cnn':
        print('\n--- Phase 1: Head training (backbone frozen, 10 epochs) ---')
        backbone.trainable = False
        model.compile(optimizer=optimizers.Adam(learning_rate=LR_PHASE1),
                      loss=losses.SparseCategoricalCrossentropy(), metrics=['accuracy'])
        h1 = model.fit(train_flow, steps_per_epoch=max(1, len(X_train)//BATCH_SIZE),
                       validation_data=(X_val, y_val), epochs=PHASE1_EPOCHS,
                       callbacks=cb, class_weight=class_weight, verbose=1)
        for k in history:
            history[k] = h1.history[k]
        print('\n--- Phase 2: Fine-tune last 30 layers (40 epochs, lr=1e-5) ---')
        backbone.trainable = True
        for layer in backbone.layers[:-30]:
            layer.trainable = False
        model.compile(optimizer=optimizers.Adam(learning_rate=LR_PHASE2),
                      loss=losses.SparseCategoricalCrossentropy(), metrics=['accuracy'])
        h2 = model.fit(train_flow, steps_per_epoch=max(1, len(X_train)//BATCH_SIZE),
                       validation_data=(X_val, y_val), epochs=EPOCHS - PHASE1_EPOCHS,
                       initial_epoch=PHASE1_EPOCHS, callbacks=cb, class_weight=class_weight, verbose=1)
        for k in history:
            history[k] = h1.history[k] + h2.history[k]
    else:
        h = model.fit(train_flow, steps_per_epoch=max(1, len(X_train)//BATCH_SIZE),
                      validation_data=(X_val, y_val), epochs=EPOCHS,
                      callbacks=cb, class_weight=class_weight, verbose=1)
        for k in history:
            history[k] = h.history[k]

    model.save(os.path.join(subdirs['models'], f'{model_name}_final.h5'))
    model.save_weights(os.path.join(subdirs['models'], f'{model_name}_final.weights.h5'))
    print('Model saved')

    y_pred_prob = model.predict(X_test, verbose=0)
    y_pred = np.argmax(y_pred_prob, axis=1)

    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred, zero_division=0)
    rec = recall_score(y_test, y_pred, zero_division=0)
    f1 = f1_score(y_test, y_pred, zero_division=0)
    cm = confusion_matrix(y_test, y_pred)
    if cm.shape == (2, 2):
        tn, fp, fn, tp = cm.ravel()
        spec = tn / (tn + fp) if (tn + fp) > 0 else 0
    else:
        spec = 0.0
    try:
        auc = roc_auc_score(y_test, y_pred_prob[:, 1])
    except:
        auc = 0.0
    print(f'  Acc: {acc:.4f} | Prec: {prec:.4f} | Rec: {rec:.4f} | Spec: {spec:.4f} | F1: {f1:.4f} | AUC: {auc:.4f}')

    results = {
        'model': model_name, 'datasets': '+'.join(dataset_names),
        'accuracy': float(acc), 'precision': float(prec), 'recall': float(rec),
        'specificity': float(spec), 'f1_score': float(f1), 'auc_roc': float(auc),
        'confusion_matrix': cm.tolist()
    }

    # Plots
    plt.figure(figsize=(5, 4))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=['Benign', 'Malignant'], yticklabels=['Benign', 'Malignant'])
    plt.title(f'{run_name} — Confusion Matrix')
    plt.ylabel('True'); plt.xlabel('Predicted'); plt.tight_layout()
    plt.savefig(os.path.join(subdirs['plots'], f'{run_name}_confusion.png'), dpi=150); plt.close()

    plt.figure(figsize=(5, 4))
    fpr_vals, tpr_vals, _ = roc_curve(y_test, y_pred_prob[:, 1])
    plt.plot(fpr_vals, tpr_vals, lw=2, label=f'AUC = {auc:.3f}')
    plt.plot([0, 1], [0, 1], 'k--', lw=1)
    plt.xlabel('FPR'); plt.ylabel('TPR'); plt.legend(loc='lower right')
    plt.title(f'{run_name} — ROC Curve'); plt.tight_layout()
    plt.savefig(os.path.join(subdirs['plots'], f'{run_name}_roc.png'), dpi=150); plt.close()

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
    ax1.plot(history['accuracy'], label='Train')
    ax1.plot(history['val_accuracy'], label='Val')
    ax1.set_title('Accuracy'); ax1.set_xlabel('Epoch'); ax1.legend(); ax1.grid(True, alpha=0.3)
    ax2.plot(history['loss'], label='Train')
    ax2.plot(history['val_loss'], label='Val')
    ax2.set_title('Loss'); ax2.set_xlabel('Epoch'); ax2.legend(); ax2.grid(True, alpha=0.3)
    plt.suptitle(run_name); plt.tight_layout()
    plt.savefig(os.path.join(subdirs['plots'], f'{run_name}_history.png'), dpi=150); plt.close()

    # Grad-CAM
    try:
        layer_name = GRAD_CAM_LAYERS.get(model_name)
        if layer_name and layer_name in [l.name for l in model.layers]:
            grad_model = models.Model(inputs=model.input,
                                      outputs=[model.get_layer(layer_name).output, model.output])
        else:
            for layer in reversed(model.layers):
                if isinstance(layer, layers.Conv2D):
                    layer_name = layer.name; break
            grad_model = models.Model(inputs=model.input,
                                      outputs=[model.get_layer(layer_name).output, model.output])
        n_samples = min(10, len(X_test))
        indices = np.random.choice(len(X_test), n_samples, replace=False)
        for idx in indices:
            img = np.expand_dims(X_test[idx], axis=0)
            with tf.GradientTape() as tape:
                conv_out, preds = grad_model(img)
                cls_idx = tf.argmax(preds[0])
                loss_val = preds[:, cls_idx]
            grads = tape.gradient(loss_val, conv_out)
            pooled = tf.reduce_mean(grads, axis=(0, 1, 2))
            heatmap = tf.reduce_sum(tf.multiply(pooled, conv_out[0]), axis=-1)
            heatmap = tf.maximum(heatmap, 0) / (tf.reduce_max(heatmap) + 1e-8)
            hmap = cv2.resize(heatmap.numpy(), (IMG_SIZE, IMG_SIZE), interpolation=cv2.INTER_CUBIC)
            hmap_colored = cv2.applyColorMap(np.uint8(255 * hmap), cv2.COLORMAP_JET)
            orig = np.uint8(X_test[idx] * 255)
            overlay = cv2.addWeighted(orig, 0.5, hmap_colored, 0.5, 0)
            cv2.imwrite(os.path.join(subdirs['gradcam'], f'{model_name}_gradcam_{idx}.png'), overlay)
        print(f'  Grad-CAM: {n_samples} samples')
    except Exception as e:
        print(f'  Grad-CAM skipped: {e}')

    return results

print('Training function defined')

In [ ]:
# Cell 8: Run all 12 experiments
all_results = {}
start_time = time_module.time()
print(f'Start: {datetime.datetime.now()}')
print(f'Architectures: {ARCHITECTURES}')
print(f'Dataset configs: {DATASET_CONFIGS}')
print(f'Total runs: {len(ARCHITECTURES) * len(DATASET_CONFIGS)}')

run_count = 0
for model_name in ARCHITECTURES:
    for ds_names in DATASET_CONFIGS:
        run_count += 1
        print(f'\n{"-"*60}')
        print(f'Run {run_count}/12: {model_name} on {ds_names}')
        print(f'{"-"*60}')
        try:
            result = train_model(model_name, ds_names)
            run_key = result['model'] + '_' + result['datasets']
            all_results[run_key] = result
            with open(os.path.join(OUTPUT_DIR, 'results_incremental.json'), 'w') as f:
                json.dump(all_results, f, indent=2, default=str)
        except Exception as e:
            import traceback
            traceback.print_exc()
            print(f'FAILED: {model_name} on {ds_names}: {e}')

elapsed = time_module.time() - start_time
print(f'\n{"="*70}')
print(f'ALL RUNS COMPLETE | Time: {elapsed/60:.1f} min ({elapsed/3600:.1f} hr)')
print(f'{"="*70}')

In [ ]:
# Cell 9: Save final results + thesis-style summary tables
with open(os.path.join(OUTPUT_DIR, 'all_results.json'), 'w') as f:
    json.dump(all_results, f, indent=2, default=str)
print(f'Results saved to {os.path.join(OUTPUT_DIR, "all_results.json")}')

# Table 8.1: Per-dataset
print('\n' + '='*100)
print('Table 8.1: Per-Dataset Performance (cf. Thesis Table 8.1)')
print('='*100)
print(f'{"Model":<15} {"Dataset":<12} {"Accuracy":<10} {"Precision":<10} {"Recall":<10} {"F1-Score":<10}')
print('-'*65)
for run_key in sorted(all_results):
    r = all_results[run_key]
    if '+' not in r['datasets']:
        print(f'{r["model"]:<15} {r["datasets"]:<12} {r["accuracy"]:<10.4f} {r["precision"]:<10.4f} {r["recall"]:<10.4f} {r["f1_score"]:<10.4f}')

# Table 6.1: Combined
print('\n' + '='*110)
print('Table 6.1: Combined Test Set Performance (cf. Thesis Table 6.1)')
print('='*110)
print(f'{"Model":<15} {"Accuracy":<10} {"Precision":<10} {"Recall":<10} {"Specificity":<12} {"F1-Score":<10} {"AUC":<10}')
print('-'*75)
for run_key in sorted(all_results):
    r = all_results[run_key]
    if '+' in r['datasets']:
        print(f'{r["model"]:<15} {r["accuracy"]:<10.4f} {r["precision"]:<10.4f} {r["recall"]:<10.4f} {r["specificity"]:<12.4f} {r["f1_score"]:<10.4f} {r["auc_roc"]:<10.4f}')
print('='*110)

print(f'\nAll outputs in: {OUTPUT_DIR}')